# Reservoir Flow Decomposition — Inflow, Outflow & Storage (German Report Style)

**Purpose:** Recreates the classic German reservoir operations plot
('Ganglinien des Gesamtzuflusses, Abgabe und Stauinhalt') showing inflow,
outflow, and reservoir volume on dual axes for the critical storm duration.

**What it does:**
- Auto-detects the critical storm duration (highest peak inflow)
- 2-panel figure: (a) T = 500yr  |  (b) T = 5000yr
- Left y-axis: inflow + outflow discharge [m³/s]
- Right y-axis: reservoir volume [Tsd m³]

**User settings:** Edit only the USER SETTINGS block at the top  
**Input:** TALSIM `.WEL` output folder (with reservoir)  
**Output:** 2-panel dual-axis PNG

---

In [ ]:
# =============================================================================
# FIGURE — Reservoir Flow Decomposition (Dual Y-axis, German report style)
# Inspired by: Ganglinien des Gesamtzuflusses, Abgabe und Stauinhalt
#
# Left  y-axis: Inflow + Outflow discharge [m³/s]
# Right y-axis: Reservoir volume [Tsd m³]
# 2-panel figure: (a) T = 500yr  |  (b) T = 5000yr
# Critical duration auto-detected.
#
# HOW TO USE:
#   Edit ONLY the USER SETTINGS block below, then run.
# =============================================================================

# %% Imports
from pathlib import Path
import re
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D

# =============================================================================
# *** USER SETTINGS — EDIT ONLY HERE ***
# =============================================================================

MAIN_FOLDER = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_100_60min")
WEL_NAME    = "Neuer_Teich_r.WEL"

COL_INFLOW  = "T001_1ZU"    # total inflow  [m³/s]
COL_OUTFLOW = "S005_1ZU"    # outflow       [m³/s]
COL_VOLUME  = "T001_VOL"    # reservoir vol [Tsd m³]

T1 = 500
T2 = 5000

# Optional: max reservoir capacity line [Tsd m³] — set None to skip
MAX_VOLUME = 430.0

OUT_DIR = Path(r"C:\Users\raah\Desktop\Project_ZR\figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Colors — matching the reference figure style
# =============================================================================
C_IN  = "#1a5276"   # deep blue  — Gesamtzufluss
C_OUT = "#c0392b"   # red        — Abgabe (outflow)
C_VOL = "#1e8449"   # green      — Stauinhalt (volume)

# =============================================================================
# Fixed settings
# =============================================================================

re_folder = re.compile(
    r"^(?P<num>\d{3})_(?P<dauer>\d+(?:[.,]\d+)?)h_(?P<yr>\d+)yr$",
    re.IGNORECASE
)
SKIP_DURATIONS = {0.25, 0.5}

# =============================================================================
# %% Helpers
# =============================================================================

def merge_split_underscore_cols(cols):
    fixed, i = [], 0
    while i < len(cols):
        if (i + 1 < len(cols)
                and cols[i + 1].startswith("_")
                and re.match(r"^[A-Za-z0-9]+$", cols[i])):
            fixed.append(cols[i] + cols[i + 1]); i += 2
        else:
            fixed.append(cols[i]); i += 1
    return fixed

_date_re = re.compile(r"^\d{2}\.\d{2}\.\d{4}$")
_time_re = re.compile(r"^\d{2}:\d{2}$")

def _find_wel(folder, wel_name):
    for name in [wel_name, wel_name.lower(), wel_name + ".txt"]:
        p = folder / name
        if p.exists():
            return p
    raise FileNotFoundError(f"WEL not found in {folder}")

def parse_wel(path, value_col):
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Datum_Zeit"):
            header_idx = i; break
    if header_idx is None:
        raise ValueError(f"Header not found in {path}")
    cols = merge_split_underscore_cols(lines[header_idx].split())
    if value_col not in cols:
        raise KeyError(
            f"'{value_col}' not found.\n"
            f"Available: {[c for c in cols if '1ZU' in c or '1AB' in c or 'VOL' in c]}"
        )
    vpos            = cols.index(value_col)
    expected_tokens = len(cols) + 1

    all_tokens = []
    for line in lines[header_idx + 1:]:
        s = line.strip()
        if not s or s.startswith("*"): continue
        toks = s.split()
        if toks and _date_re.match(toks[0]):
            all_tokens.extend(toks)

    times, values = [], []
    i = 0
    n = len(all_tokens)
    while i < n - expected_tokens + 1:
        if not (_date_re.match(all_tokens[i]) and _time_re.match(all_tokens[i + 1])):
            i += 1; continue
        if i + expected_tokens > n: break
        rec = all_tokens[i: i + expected_tokens]
        try:
            ts = dt.datetime.strptime(rec[0] + " " + rec[1], "%d.%m.%Y %H:%M")
        except Exception:
            i += 1; continue
        try:
            val = float(rec[vpos + 1].replace(",", "."))
        except Exception:
            i += expected_tokens; continue
        times.append(ts); values.append(val)
        i += expected_tokens

    q   = np.asarray(values, dtype=float)
    t_h = np.array([(t - times[0]).total_seconds() / 3600
                    for t in times]) if times else np.arange(len(q), dtype=float)
    return {"q": q, "t_h": t_h}


def load_critical(root, wel_name, col_in, col_out, col_vol, return_period):
    """Load critical duration — highest inflow peak — with all 3 series."""
    candidates = {}
    for sf in sorted(root.iterdir()):
        if not sf.is_dir(): continue
        m = re_folder.match(sf.name)
        if not m: continue
        if int(m.group("yr")) != return_period: continue
        dauer_h = float(m.group("dauer").replace(",", "."))
        if dauer_h in SKIP_DURATIONS: continue
        try:
            wel    = _find_wel(sf, wel_name)
            p_in   = parse_wel(wel, col_in)
            p_out  = parse_wel(wel, col_out)
            p_vol  = parse_wel(wel, col_vol)
            if len(p_in["q"]) > 0:
                candidates[dauer_h] = {
                    "q_in":    p_in["q"],   "t_in":  p_in["t_h"],
                    "q_out":   p_out["q"],  "t_out": p_out["t_h"],
                    "q_vol":   p_vol["q"],  "t_vol": p_vol["t_h"],
                    "qmax_in": float(np.nanmax(p_in["q"])),
                }
        except Exception as e:
            print(f"  [SKIP] {sf.name}: {e}")

    if not candidates:
        print(f"  [WARNING] No data for T={return_period}yr"); return None

    d = max(candidates, key=lambda x: candidates[x]["qmax_in"])
    r = candidates[d]
    r["dauer_h"]   = d
    r["pct_att"]   = 100 * (np.nanmax(r["q_in"]) - np.nanmax(r["q_out"])) / np.nanmax(r["q_in"])
    r["peak_delay"]= (r["t_out"][np.nanargmax(r["q_out"])] -
                      r["t_in"][np.nanargmax(r["q_in"])])
    return r


# =============================================================================
# %% Panel plotter — dual y-axis
# =============================================================================

def plot_panel(ax_q, data, panel_title):
    """
    Left axis (ax_q)  : Inflow + Outflow [m³/s]
    Right axis (ax_v) : Reservoir volume [Tsd m³]
    """
    if data is None:
        ax_q.set_title(f"{panel_title}\n(no data)", pad=7)
        return

    # Create right y-axis twin
    ax_v = ax_q.twinx()

    t_in  = data["t_in"]
    q_in  = data["q_in"]
    t_out = data["t_out"]
    q_out = data["q_out"]
    t_vol = data["t_vol"]
    q_vol = data["q_vol"]
    d     = data["dauer_h"]

    qmax  = max(float(np.nanmax(q_in)), float(np.nanmax(q_out)))
    vmax  = float(np.nanmax(q_vol))
    tmax  = max(float(t_in[-1]), float(t_out[-1]), float(t_vol[-1])) * 1.02

    # ── Left axis: discharge lines ────────────────────────────────────────────
    ax_q.plot(t_in,  q_in,  color=C_IN,  lw=1.0, ls="-",  zorder=4,
              label=f"Inflow  ({COL_INFLOW})")
    ax_q.plot(t_out, q_out, color=C_OUT, lw=1.0, ls="-",  zorder=4,
              label=f"Outflow  ({COL_OUTFLOW})")

    # Peak markers on discharge
    for q_arr, t_arr, color in [(q_in, t_in, C_IN), (q_out, t_out, C_OUT)]:
        idx = int(np.nanargmax(q_arr))
        ax_q.plot(t_arr[idx], q_arr[idx], "^", ms=6,
                  color=color, markeredgewidth=0.5,
                  markeredgecolor="white", zorder=10)

    # ── Right axis: volume line ───────────────────────────────────────────────
    ax_v.plot(t_vol, q_vol, color=C_VOL, lw=1.0, ls="-", zorder=3,
              label=f"Volume  ({COL_VOLUME})")

    # Max volume capacity line (right axis)
    if MAX_VOLUME is not None:
        # Capacity dashed line
        ax_v.axhline(MAX_VOLUME, color="0.40", lw=0.8, ls="--",
                     zorder=3, label=f"Max capacity = {MAX_VOLUME:g} Tsd m³")
        ax_v.text(tmax * 0.98, MAX_VOLUME,
                  f" {MAX_VOLUME:g} Tsd m³",
                  fontsize=6.5, color="0.35",
                  va="bottom", ha="right", style="italic")

    # ── Axis limits and formatting ────────────────────────────────────────────
    ax_q.set_xlim(0, tmax)
    ax_q.set_ylim(0, qmax * 1.20)
    ax_q.xaxis.set_major_locator(MaxNLocator(8, integer=False))
    ax_q.tick_params(axis="both", length=4)
    ax_q.set_xlabel("Time [h] (from event start)", fontsize=8)
    ax_q.set_ylabel("Discharge  Q [m³/s]", fontsize=8, color="0.2")
    ax_q.set_title(panel_title, pad=7, fontsize=9, loc="left")
    ax_q.spines["top"].set_visible(False)
    ax_q.grid(True, linestyle="--", alpha=0.30, zorder=0)
    ax_q.set_axisbelow(True)

    # Volume axis starts at minimum volume (initial state), not zero
    vmin = float(np.nanmin(q_vol))
    ax_v.set_ylim(vmin - (vmax - vmin) * 0.05,
                  max(vmax, MAX_VOLUME if MAX_VOLUME else vmax) * 1.12)
    ax_v.set_ylabel("Volume  [Tsd m³]", fontsize=8, color=C_VOL)
    ax_v.tick_params(axis="y", colors=C_VOL, length=4)
    ax_v.spines["top"].set_visible(False)
    ax_v.spines["right"].set_edgecolor(C_VOL)

    # ── Metrics annotation ────────────────────────────────────────────────────
    info = (
        f"d = {d:g} h\n"
        f"Qmax in  = {np.nanmax(q_in):.2f} m³/s\n"
        f"Qmax out = {np.nanmax(q_out):.2f} m³/s\n"
        f"Attenuat.= {data['pct_att']:.1f} %\n"
        f"Delay    = {data['peak_delay']:.1f} h\n"
        f"Vmax     = {vmax:.1f} Tsd m³"
    )
    # Box anchored at top-right, text left-aligned inside
    ax_q.text(0.98, 0.97, info,
              transform=ax_q.transAxes,
              fontsize=6.2, ha="right", va="top",
              family="monospace", color="0.15",
              multialignment="left",
              bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                        edgecolor="0.55", alpha=0.95, linewidth=0.7),
              zorder=20)

    return ax_v


# =============================================================================
# %% Main
# =============================================================================

plt.rcParams.update({
    "font.family":    "monospace",
    "font.size":       8,
    "axes.titlesize":  9,
    "axes.labelsize":  8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi":      150,
})

print("=" * 60)
print("Reservoir Flow Decomposition — Dual Y-axis")
print("=" * 60)

print(f"\nLoading T = {T1} yr ...")
data_T1 = load_critical(MAIN_FOLDER, WEL_NAME,
                         COL_INFLOW, COL_OUTFLOW, COL_VOLUME, T1)
if data_T1:
    print(f"  d={data_T1['dauer_h']:g}h  Qin={np.nanmax(data_T1['q_in']):.2f}  "
          f"Qout={np.nanmax(data_T1['q_out']):.2f}  att={data_T1['pct_att']:.1f}%")

print(f"\nLoading T = {T2} yr ...")
data_T2 = load_critical(MAIN_FOLDER, WEL_NAME,
                         COL_INFLOW, COL_OUTFLOW, COL_VOLUME, T2)
if data_T2:
    print(f"  d={data_T2['dauer_h']:g}h  Qin={np.nanmax(data_T2['q_in']):.2f}  "
          f"Qout={np.nanmax(data_T2['q_out']):.2f}  att={data_T2['pct_att']:.1f}%")

# 1 row × 2 cols
# Row-wise layout: T=500yr top, T=5000yr bottom
# Each panel is full-width → wider view of the hydrograph shape
FIG_WIDTH  = 6.30
FIG_HEIGHT = 7.00   # 2 rows — same height budget as other 2-row figures
fig, axes  = plt.subplots(2, 1, figsize=(FIG_WIDTH, FIG_HEIGHT))

print("\nPlotting...")
ax_v1 = plot_panel(axes[0], data_T1, f"(a)  T = {T1} yr")
ax_v2 = plot_panel(axes[1], data_T2, f"(b)  T = {T2} yr")

# ── Shared legend — collect from both axes ────────────────────────────────────
legend_elements = [
    Line2D([0], [0], color=C_IN,  lw=1.8, ls="-",
           label=f"Inflow"),
    Line2D([0], [0], color=C_OUT, lw=1.8, ls="-",
           label=f"Outflow"),
    Line2D([0], [0], color=C_VOL, lw=1.8, ls="-",
           label=f"Volume"),
]
if MAX_VOLUME is not None:
    legend_elements.append(
        Line2D([0], [0], color="0.45", lw=1.0, ls="--",
               label=f"Max capacity")
    )

fig.tight_layout()
fig.subplots_adjust(hspace=0.42, bottom=0.14, right=0.88)
fig.legend(
    handles        = legend_elements,
    loc            = "lower center",
    ncol           = 4,
    fontsize       = 7,
    framealpha     = 0.92,
    edgecolor      = "0.7",
    borderpad      = 0.5,
    handlelength   = 1.8,
    columnspacing  = 1.5,
    bbox_to_anchor = (0.44, 0.01),
)

outpng = OUT_DIR / "Fig_FlowDecomposition_DualAxis.png"
fig.savefig(outpng, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"\nSaved: {outpng}")
print("In Word: Insert > Pictures > set width = 16.0 cm")

# =============================================================================
# %% COLUMN HELPER — uncomment to check available columns
# =============================================================================
#
# for sf in sorted(MAIN_FOLDER.iterdir()):
#     if sf.is_dir():
#         wel = sf / WEL_NAME
#         if wel.exists():
#             lines = wel.read_text(encoding="utf-8", errors="replace").splitlines()
#             for line in lines:
#                 if line.strip().startswith("Datum_Zeit"):
#                     cols = line.split()
#                     print("1ZU:", [c for c in cols if "1ZU" in c])
#                     print("VOL:", [c for c in cols if "VOL" in c])
#                     break
#         break